# P4 Notebook-First Master — observed-development

| 항목 | 내용 |
|---|---|
| 목적 | 23개 Agent Notebook의 source gate, 실행 순서, stage manifest, 최종 bundle QA를 통합한다. |
| 담당 Agent | `P4-A3-CONTROL` |
| Stage ID | `A3-MASTER` |
| 입력 | stage registry, Agent 1·2·4 source Notebook, observed input과 NCS handoff |
| 처리 | 구조감사 → architecture/index → fresh-kernel child 실행 → manifest 수집 → bundle gate |
| 출력 | executed Notebook, 실행결과 CSV, 4개 종료 artifact, 최종 보고 입력 |
| 선행 Gate | `CRAWL_OBSERVED_INPUT_READY`, `OBSERVED_PARSE_READY`, `NCS_MAPPING_DEV_READY` |
| 후속 활용 | 사용자 검수용 Notebook bundle; empirical analysis나 production 승격에는 사용하지 않는다. |


In [1]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A3-CONTROL"
STAGE_ID = "A3-MASTER"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "notebook-bundle-v1"
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "crawl/control/NOTEBOOK_STAGE_REGISTRY.yaml"
OUTPUT_ROOT = "crawl/runs/notebooks/observed-dev/MASTER_20260806_01/master_stage"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False

# Injected into the executed copy by crawl.control.notebook_bundle
RUN_MODE = 'observed-dev'
OUTPUT_ROOT = 'crawl/runs/notebooks/observed-dev/MASTER_20260806_01/stages/crawl/notebooks/P4_Notebook_First_Master'
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False

In [2]:
import platform
import sys
from pathlib import Path

from crawl.control.notebook_bundle import (
    NOTEBOOKS,
    audit_source_bundle,
    collect_stage_manifests,
    execute_notebook_plan,
    git_head,
    load_stage_registry,
    repository_architecture,
    resolve_project_root,
    sha256_file,
    write_master_stage_artifacts,
)

PROJECT_ROOT = resolve_project_root(Path.cwd())
CONTRACT_PATH = PROJECT_ROOT / "crawl/control/NOTEBOOK_EXECUTION_CONTRACT.md"
environment = {
    "repoRoot": str(PROJECT_ROOT),
    "gitHead": git_head(PROJECT_ROOT),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "contractPath": str(CONTRACT_PATH.relative_to(PROJECT_ROOT)),
    "contractSha256": sha256_file(CONTRACT_PATH),
}
environment


{'repoRoot': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/project_4',
 'gitHead': '71edc867a8e869b78340e4fe253ced9312894077',
 'python': '3.12.3',
 'platform': 'Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'contractPath': 'crawl/control/NOTEBOOK_EXECUTION_CONTRACT.md',
 'contractSha256': '1fa796ef6f14896bcd542ce156a5063055fe9d351f0b1691cda346af672f0ae4'}

In [3]:
registry = load_stage_registry(PROJECT_ROOT)
source_audit = audit_source_bundle(PROJECT_ROOT)
invalid_sources = [row for row in source_audit if not row.get("valid")]
input_audit = {
    "registryVersion": registry["registryVersion"],
    "contractVersion": registry["contractVersion"],
    "sourceNotebookCount": len(source_audit),
    "validSourceNotebookCount": len(source_audit) - len(invalid_sources),
    "dataProvenance": "OBSERVED_DEVELOPMENT_ONLY",
    "empiricalAnalysisAllowed": EMPIRICAL_ANALYSIS_ALLOWED,
    "upstreamGate": "OBSERVED_DEV_CSV_READY",
}
if FAIL_ON_GATE and invalid_sources:
    raise RuntimeError(f"Notebook source gate failed: {invalid_sources}")
input_audit


{'registryVersion': 'notebook-stage-registry-v1',
 'contractVersion': '2.1.2',
 'sourceNotebookCount': 24,
 'validSourceNotebookCount': 24,
 'dataProvenance': 'OBSERVED_DEVELOPMENT_ONLY',
 'empiricalAnalysisAllowed': False,
 'upstreamGate': 'OBSERVED_DEV_CSV_READY'}

## 1부 — Source·수집·전처리·NCS

Agent 1은 offline/dry-run/replay 수집 단계를, Agent 2는 observed 전처리 단계를,
Agent 4는 lexical baseline 단계를 담당한다.

## 2부 — 통합 실행·검수

Master는 stage registry 순서로 source Notebook을 fresh kernel에서 실행하고,
실행본과 stage artifact를 source와 분리해 보존한다.


In [4]:
architecture_rows = repository_architecture(PROJECT_ROOT)
architecture_summary = {
    "agent1": sum(row["agentId"] == "P4-A1-SOURCE" for row in architecture_rows),
    "agent2": sum(row["agentId"] == "P4-A2-PIPELINE" for row in architecture_rows),
    "agent4": sum(row["agentId"] == "P4-A4-NCS" for row in architecture_rows),
    "master": sum(row["agentId"] == "P4-A3-CONTROL" for row in architecture_rows),
    "total": len(architecture_rows),
}
architecture_summary, architecture_rows


({'agent1': 5, 'agent2': 12, 'agent4': 6, 'master': 1, 'total': 24},
 [{'agentId': 'P4-A1-SOURCE',
   'stageId': 'A1-00-RECOVER',
   'notebook': 'crawl/notebooks/00RecoverSourceState.ipynb',
   'exists': True,
   'bytes': 8337},
  {'agentId': 'P4-A1-SOURCE',
   'stageId': 'A1-01-INDEX',
   'notebook': 'crawl/notebooks/01CollectLinkareerIndex.ipynb',
   'exists': True,
   'bytes': 7936},
  {'agentId': 'P4-A1-SOURCE',
   'stageId': 'A1-02-DETAIL',
   'notebook': 'crawl/notebooks/02CollectPostingDetail.ipynb',
   'exists': True,
   'bytes': 8387},
  {'agentId': 'P4-A1-SOURCE',
   'stageId': 'A1-03-ASSET',
   'notebook': 'crawl/notebooks/03CollectPostingAssets.ipynb',
   'exists': True,
   'bytes': 8459},
  {'agentId': 'P4-A1-SOURCE',
   'stageId': 'A1-04-RELEASE',
   'notebook': 'crawl/notebooks/04BuildCrawlRelease.ipynb',
   'exists': True,
   'bytes': 8914},
  {'agentId': 'P4-A2-PIPELINE',
   'stageId': 'A2-00-CONTRACT-AUDIT',
   'notebook': 'pipeline/notebooks/00ContractAndInputAudit.i

In [5]:
registry_rows = [
    {
        "stageId": stage["stageId"],
        "ownerAgent": stage["ownerAgent"],
        "notebookPath": stage["notebookPath"],
        "requiredGate": stage["requiredGate"],
        "producedGate": stage["producedGate"],
    }
    for stage in registry["stages"]
]
registry_rows


[{'stageId': 'A1-00-RECOVER',
  'ownerAgent': 'P4-A1-SOURCE',
  'notebookPath': 'crawl/notebooks/00RecoverSourceState.ipynb',
  'requiredGate': 'NOT_EVALUATED',
  'producedGate': 'SOURCE_POLICY_READY'},
 {'stageId': 'A1-01-INDEX',
  'ownerAgent': 'P4-A1-SOURCE',
  'notebookPath': 'crawl/notebooks/01CollectLinkareerIndex.ipynb',
  'requiredGate': 'SOURCE_POLICY_READY',
  'producedGate': 'CRAWL_INDEX_OBSERVED_READY'},
 {'stageId': 'A1-02-DETAIL',
  'ownerAgent': 'P4-A1-SOURCE',
  'notebookPath': 'crawl/notebooks/02CollectPostingDetail.ipynb',
  'requiredGate': 'CRAWL_INDEX_OBSERVED_READY',
  'producedGate': 'CRAWL_OBSERVED_INPUT_READY'},
 {'stageId': 'A1-03-ASSET',
  'ownerAgent': 'P4-A1-SOURCE',
  'notebookPath': 'crawl/notebooks/03CollectPostingAssets.ipynb',
  'requiredGate': 'CRAWL_OBSERVED_INPUT_READY',
  'producedGate': 'CRAWL_ASSET_LINEAGE_READY'},
 {'stageId': 'A1-04-RELEASE',
  'ownerAgent': 'P4-A1-SOURCE',
  'notebookPath': 'crawl/notebooks/04BuildCrawlRelease.ipynb',
  'requir

In [6]:
MASTER_OUTPUT = (PROJECT_ROOT / OUTPUT_ROOT).resolve() if not Path(OUTPUT_ROOT).is_absolute() else Path(OUTPUT_ROOT)
CHILD_RUN_ROOT = MASTER_OUTPUT.parent / "children"
execution_results = execute_notebook_plan(
    PROJECT_ROOT,
    CHILD_RUN_ROOT,
    include_master=False,
    fail_fast=FAIL_ON_GATE,
)
failed = [row for row in execution_results if row["status"] != "PASS"]
if FAIL_ON_GATE and failed:
    raise RuntimeError(f"Child Notebook execution failed: {failed[0]}")
execution_results


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


...

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[{'notebook': 'crawl/notebooks/00RecoverSourceState.ipynb',
  'status': 'PASS',
  'elapsedSeconds': 1.559,
  'sourceSha256': '4bf405ece28246d893ea347ee1aeb27a4283f25a98926473c1fdcf6aac37f3c9',
  'executedPath': 'crawl/runs/notebooks/observed-dev/MASTER_20260806_01/stages/crawl/notebooks/children/executed/crawl/notebooks/00RecoverSourceState.ipynb',
  'executedSha256': '79b735d7ca3fc8177c63671c87fc47e912a46e28b1fedea17dcdd186cef89150',
  'executedOutputs': 2,
  'stageOutputRoot': 'crawl/runs/notebooks/observed-dev/MASTER_20260806_01/stages/crawl/notebooks/children/agent_runs/AGENT1/A1-00-RECOVER',
  'error': '',
  'agentId': 'P4-A1-SOURCE',
  'stageId': 'A1-00-RECOVER'},
 {'notebook': 'crawl/notebooks/01CollectLinkareerIndex.ipynb',
  'status': 'PASS',
  'elapsedSeconds': 1.373,
  'sourceSha256': '01d65fdb65d5610f437d0fbd06f5d692b5122d070fddb6be06ab41cfc1293806',
  'executedPath': 'crawl/runs/notebooks/observed-dev/MASTER_20260806_01/stages/crawl/notebooks/children/executed/crawl/notebo

In [7]:
stage_manifests = collect_stage_manifests(CHILD_RUN_ROOT)
manifest_summary = {
    "collected": len(stage_manifests),
    "succeeded": sum(row.get("status") == "SUCCEEDED" for row in stage_manifests),
    "notEvaluated": sum(row.get("status") == "NOT_EVALUATED" for row in stage_manifests),
    "failed": sum(row.get("status") in {"FAILED", "INVALID"} for row in stage_manifests),
}
manifest_summary


{'collected': 27, 'succeeded': 25, 'notEvaluated': 1, 'failed': 1}

In [8]:
final_export = PROJECT_ROOT / "crawl/data/exports/observed-dev/OBSERVED_DEV_20260806_01"
required_review_files = [
    "posting_normalized.csv", "posting_tracks.csv", "posting_sections.csv",
    "requirement_facts.csv", "career_access_labels.csv", "posting_ncs_candidates.csv",
    "preprocessed_posting_tracks.csv", "data_quality_summary.csv", "CHECKSUMS.sha256",
]
bundle_qa = {
    "requiredReviewFiles": len(required_review_files),
    "presentReviewFiles": sum((final_export / name).is_file() for name in required_review_files),
    "childExecutionPass": sum(row["status"] == "PASS" for row in execution_results),
    "childExecutionTotal": len(execution_results),
    "sourceOutputCount": sum(int(row.get("sourceOutputs", 0)) for row in source_audit),
    "empiricalAnalysisAllowed": False,
    "promotionAllowed": False,
}
if FAIL_ON_GATE and (
    bundle_qa["presentReviewFiles"] != bundle_qa["requiredReviewFiles"]
    or bundle_qa["childExecutionPass"] != bundle_qa["childExecutionTotal"]
    or bundle_qa["sourceOutputCount"] != 0
):
    raise RuntimeError(f"Master bundle QA failed: {bundle_qa}")
bundle_qa


{'requiredReviewFiles': 9,
 'presentReviewFiles': 9,
 'childExecutionPass': 23,
 'childExecutionTotal': 23,
 'sourceOutputCount': 0,
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False}

In [9]:
termination = write_master_stage_artifacts(
    MASTER_OUTPUT,
    PROJECT_ROOT,
    source_audit,
    execution_results,
    stage_manifests,
)
{
    "status": "NOTEBOOK_EXECUTION_READY_OBSERVED_DEV",
    "terminationArtifacts": termination,
    "reviewCsv": str((final_export / "preprocessed_posting_tracks.csv").relative_to(PROJECT_ROOT)),
    "forbidden": ["CRAWL_RELEASE_READY", "DATA_READY_RQ1_RQ2A", "DATA_READY_RQ2B", "ANALYSIS_READY"],
}


{'status': 'NOTEBOOK_EXECUTION_READY_OBSERVED_DEV',
 'terminationArtifacts': {'stage_manifest.json': '55672097304ffb854423c43436cbfbfbec46113e11cd82d14c065a632b678a63',
  'stage_metrics.json': 'a57fc401d50581391eb8aa3605fe6279d0661ffd6f54f33e9eb6a08c24509aa9',
  'stage_quality.csv': '426a6f57de19bf8e3c7e3e32137147631023c7af58700ef1eebb1c360fdb28bc',
  'CHECKSUMS.sha256': '49e236bda63a368604782c33c0cacdd4cfb88b4c11c7f2d9a9f4f0fc06f5610c'},
 'reviewCsv': 'crawl/data/exports/observed-dev/OBSERVED_DEV_20260806_01/preprocessed_posting_tracks.csv',
 'forbidden': ['CRAWL_RELEASE_READY',
  'DATA_READY_RQ1_RQ2A',
  'DATA_READY_RQ2B',
  'ANALYSIS_READY']}